# Part 4-  Model Distillation / Quantization

**Goal:** compress our best-performing fine-tuned BERT classifier and quantify the speed / memory / carbon vs. performance trade-off, following the Pruna-based approach from the professor's *Session 7.3* notebook.


In [2]:
import os, sys, gc, json, random, copy
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from utils import preprocess_for_bert, measure_inference_metrics  
from Part_2_BERT import train_bert, tokenizer, max_length     
import matplotlib.pyplot as plt
import setup

os.getcwd()
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

## load data with load and save function
data_path = Path("../data/trustpilot_reviews.pkl")
if data_path.exists():
    df = pd.read_pickle(data_path)
else:
    df = setup.load_dataset_and_save_as_csv() 
df.head()


,category,company,description,title,review,stars
0,Animals & Pets,ruffandtumbledogcoats.com,At Ruff and Tumble we are proud to be the mark...,Great quality dog drying robe although…,Great quality dog drying robe although had to ...,5
1,Animals & Pets,ruffandtumbledogcoats.com,At Ruff and Tumble we are proud to be the mark...,Really prompt service,"Really prompt service, The sofa covers have no...",5
2,Animals & Pets,ruffandtumbledogcoats.com,At Ruff and Tumble we are proud to be the mark...,Life saver,I’ve purchased first of those coats in May2020...,5
3,Animals & Pets,ruffandtumbledogcoats.com,At Ruff and Tumble we are proud to be the mark...,Brilliant coats,Brilliant coats. Really like the limited editi...,5
4,Animals & Pets,ruffandtumbledogcoats.com,At Ruff and Tumble we are proud to be the mark...,Great company and products,Great company and products. This is my 3rd dry...,5


In [3]:

SEED = 123
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(SEED)



In [4]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "scripts":
    PROJECT_ROOT = PROJECT_ROOT.parent

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Project root : {PROJECT_ROOT}")
print(f"Device       : {device}")
print(f"Torch        : {torch.__version__}")
print(f"Tokenizer    : bert-base-uncased | max_length={max_length}")

# output dirs
RESULTS_DIR = PROJECT_ROOT / "results"; RESULTS_DIR.mkdir(exist_ok=True)
MODELS_DIR  = PROJECT_ROOT / "models";  MODELS_DIR.mkdir(exist_ok=True)
FIGURES_DIR = PROJECT_ROOT / "figures"; FIGURES_DIR.mkdir(exist_ok=True)

Project root : c:\Users\majoa\OneDrive\Desktop\Learning\Grad_School\advanced_nlp\advanced_nlp_final
Device       : cpu
Torch        : 2.12.0+cpu
Tokenizer    : bert-base-uncased | max_length=64


### 1.3 Load data with the *same* test split as Parts 1–3

In [5]:
CATEGORIES = ["Travel & Vacation", "Media & Publishing"]

df = pd.read_pickle(PROJECT_ROOT / "data" / "trustpilot_reviews.pkl")
df = df[df["category"].isin(CATEGORIES)]

# Replicate setup_part3.cut_and_save_data(filtered_df, 100): full reshuffle, seed 42.
df = df.sample(n=len(df), random_state=SEED).reset_index(drop=True)

# Same preprocessing + labeling as Part 3.
df["review"] = df["review"].apply(preprocess_for_bert)
labeled_df = df.dropna(subset=["stars", "review"])
texts  = labeled_df["review"].tolist()
labels = (labeled_df["stars"].astype(int) - 1).tolist()   # 0-4 for 5 classes
num_labels = 5

set_seed(SEED)
train_texts, valid_texts, train_labels, valid_labels = train_test_split(
    texts, labels, test_size=0.2, stratify=labels, random_state=SEED
)

print(f"Total labeled : {len(texts)}")
print(f"Train         : {len(train_texts)}")
print(f"Test (valid)  : {len(valid_texts)}")
print("Test label distribution (0-4):")
print(pd.Series(valid_labels).value_counts().sort_index())

Total labeled : 10776
Train         : 8620
Test (valid)  : 2156
Test label distribution (0-4):
0    436
1    402
2    415
3    441
4    462
Name: count, dtype: int64


In [6]:
from transformers import AutoModelForSequenceClassification

TEACHER_DIR = MODELS_DIR / "bert_teacher_100"
TEACHER_EPOCHS = 3          # raise to ~10 on GPU for a stronger teacher
RETRAIN = False             # set True to force retraining even if a saved teacher exists

if TEACHER_DIR.exists() and not RETRAIN:
    print(f"Loading existing teacher from {TEACHER_DIR}")
    teacher = AutoModelForSequenceClassification.from_pretrained(TEACHER_DIR)
else:
    print(f"Training teacher (BERT @ 100% data, {TEACHER_EPOCHS} epochs)...")
    set_seed(SEED)
    teacher, _, _ = train_bert(
        train_texts, train_labels, valid_texts, valid_labels,
        num_train_epochs=TEACHER_EPOCHS,
        num_labels=num_labels,
        early_stopping=True,                 # train split is > 500 examples
        output_dir=str(PROJECT_ROOT / "tmp_bert_teacher"),
    )
    teacher.save_pretrained(TEACHER_DIR)
    tokenizer.save_pretrained(TEACHER_DIR)
    print(f"Saved teacher to {TEACHER_DIR}")

teacher.to(device).eval()
n_params = sum(p.numel() for p in teacher.parameters())
print(f"Teacher parameters: {n_params:,}")

Loading existing teacher from c:\Users\majoa\OneDrive\Desktop\Learning\Grad_School\advanced_nlp\advanced_nlp_final\models\bert_teacher_100


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Teacher parameters: 109,486,085




Part 3 trained the 100%-data model in a loop and deleted the checkpoint, so nothing was persisted. We retrain it here once via the shared `train_bert` (identical architecture,
tokenizer, and hyper-parameters as Part 3) and **save it** to `models/bert_teacher_100/`
so the rest of Part 4 — and any re-run — can load it directly.

`load_best_model_at_end=True` inside `train_bert` means the returned model is the best
checkpoint by validation loss.



measure_inference_metrics (from `utils.py` from the class notebook) iterates the dataset in fixed-size batches and stacks each batch into tensors, so the test set must be tokenized to a **fixed length**
(padded and consistent with what we've done in other notebooks) and exposed with the columns `input_ids`, `attention_mask`, `label`.

We pad to the same `max_length=64` the model was trained with.

In [7]:
from datasets import Dataset as HFDataset

def build_eval_dataset(texts, labels):
    ds = HFDataset.from_dict({"text": list(texts), "label": list(labels)})
    def tok(batch):
        return tokenizer(batch["text"], padding="max_length",
                         truncation=True, max_length=max_length)
    ds = ds.map(tok, batched=True)
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    return ds

test_ds = build_eval_dataset(valid_texts, valid_labels)
print(test_ds)
print("Sanity — one batch shapes:")
b = test_ds[0:4]
print(" input_ids     :", b["input_ids"].shape)
print(" attention_mask:", b["attention_mask"].shape)
print(" label         :", b["label"].shape)

Map:   0%|          | 0/2156 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2156
})
Sanity — one batch shapes:
 input_ids     : torch.Size([4, 64])
 attention_mask: torch.Size([4, 64])
 label         : torch.Size([4])


We run `measure_inference_metrics` on the teacher to capture the full baseline: (1) F1 (2) precision (3) recall (macro), (4) inference speed (samples/sec), RAM, GPU memory (only populated when CUDA is available), and carbon footprint via codecarbon (consistent with class notebook)

In [8]:
BATCH_SIZE = 32

print("Evaluating ORIGINAL (teacher) model...")
baseline_metrics = measure_inference_metrics(
    teacher, test_ds, device=device, batch_size=BATCH_SIZE
)
baseline_metrics["n_params"] = int(n_params)
baseline_metrics["n_test_samples"] = len(test_ds)

print("\n" + "=" * 52)
print("ORIGINAL BERT — BASELINE METRICS")
print("=" * 52)
print("\nClassification (macro):")
for k in ("f1_macro", "precision_macro", "recall_macro"):
    print(f"  {k:18s}: {baseline_metrics[k]:.4f}")
print("\nEfficiency:")
print(f"  inference_speed   : {baseline_metrics['inference_speed (samples/sec)']:.2f} samples/sec")
print(f"  cpu_memory_peak   : {baseline_metrics['cpu_memory_peak (MB)']:.1f} MB")
if 'gpu_memory_used (MB)' in baseline_metrics:
    print(f"  gpu_memory_used   : {baseline_metrics['gpu_memory_used (MB)']:.1f} MB")
print(f"  carbon_footprint  : {baseline_metrics['carbon_footprint (kg CO2eq)']:.8f} kg CO2eq")
print(f"  parameters        : {baseline_metrics['n_params']:,}")

# Persist so later steps / kernel restarts don't lose the baseline.
results = {"original": baseline_metrics}
with open(RESULTS_DIR / "part4_baseline.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"\nSaved baseline to {RESULTS_DIR / 'part4_baseline.json'}")

Evaluating ORIGINAL (teacher) model...


100%|██████████| 68/68 [02:20<00:00,  2.06s/it]



ORIGINAL BERT — BASELINE METRICS

Classification (macro):
  f1_macro          : 0.5494
  precision_macro   : 0.5467
  recall_macro      : 0.5559

Efficiency:
  inference_speed   : 15.36 samples/sec
  cpu_memory_peak   : 1164.1 MB
  carbon_footprint  : 0.00009762 kg CO2eq
  parameters        : 109,486,085

Saved baseline to c:\Users\majoa\OneDrive\Desktop\Learning\Grad_School\advanced_nlp\advanced_nlp_final\results\part4_baseline.json


## Same evaluation method

One wrapper around the metrics function so every variant (baseline + each compressed model) is benchmarked **identically** on the same test set, recorded into a single "results" dict, and saved.

In [ ]:
import tempfile, shutil

RESULTS_FILE = RESULTS_DIR / "part4_compression_results.json"

## cumulative results dict, seeded with the Step 1 baseline
results = {"original": baseline_metrics}

def count_params(model):
    return sum(p.numel() for p in model.parameters())

def model_size_mb(model):
    """On-disk size (MB) via save_pretrained; None if the model can't be serialized."""
    tmp = tempfile.mkdtemp()
    try:
        model.save_pretrained(tmp)
        total = sum(
            os.path.getsize(os.path.join(root, f))
            for root, _, files in os.walk(tmp) for f in files
        )
        return total / (1024 * 1024)
    except Exception:
        return None
    finally:
        shutil.rmtree(tmp, ignore_errors=True)

def evaluate_variant(model, name, batch_size=BATCH_SIZE, save=True):
    """Benchmark one model variant on the shared test set and record it."""
    ## same function, same test_ds, same batch size for every variant
    m = measure_inference_metrics(model, test_ds, device=device, batch_size=batch_size)
    m["n_params"] = int(count_params(model))
    m["size_mb"] = model_size_mb(model)          # measured while the model is still alive
    m["n_test_samples"] = len(test_ds)
    results[name] = m

    if save:
        with open(RESULTS_FILE, "w") as f:
            json.dump(results, f, indent=2)

    ## compact, consistent summary
    size_str = f"{m['size_mb']:.1f} MB" if m["size_mb"] is not None else "n/a"
    print(f"[{name}]")
    print(f"  accuracy          : {m['accuracy']:.4f}")
    print(f"  f1_macro          : {m['f1_macro']:.4f}")
    print(f"  precision_macro   : {m['precision_macro']:.4f}")
    print(f"  recall_macro      : {m['recall_macro']:.4f}")
    print(f"  inference_speed   : {m['inference_speed (samples/sec)']:.2f} samples/sec")
    print(f"  cpu_memory_peak   : {m['cpu_memory_peak (MB)']:.1f} MB")
    if "gpu_memory_used (MB)" in m:
        print(f"  gpu_memory_used   : {m['gpu_memory_used (MB)']:.1f} MB")
    print(f"  carbon_footprint  : {m['carbon_footprint (kg CO2eq)']:.8f} kg CO2eq")
    print(f"  parameters        : {m['n_params']:,}")
    print(f"  size_on_disk      : {size_str}")
    return m

## give the baseline its on-disk size too (teacher is still in memory at this point)
baseline_metrics.setdefault("size_mb", model_size_mb(teacher))

## persist the baseline through the same path the variants will use
with open(RESULTS_FILE, "w") as f:
    json.dump(results, f, indent=2)
print(f"Harness ready. Baseline recorded as 'original'. Saving to {RESULTS_FILE.name}")

**Define Pruna compression strategies**

Mirror professor's notebook on pruna configurations

In [ ]:
## each strategy: name, the SmashConfig algo dict, GPU requirement flag, short description
## applied in Step 4 on a fresh deepcopy of the teacher
compression_strategies = [
    {
        "name": "unstructured_pruning",
        "config": {"pruner": "torch_unstructured"},
        "needs_cuda": False,
        "description": "Unstructured magnitude pruning (zeros individual weights)",
    },
        # {
        # "name": "half_precision", ### This will crash as it will convert all values, including indices
        # "config_fn": lambda: {    ## it is meant to reduce the quantities (weights, activations, logits),
        #     "quantizer": "half"    ## but it impacts all numbers/quantities, including token index values.
        # },
        # "requires": [],
        # "description": "Half precision (FP16) quantization"
    # },
    {
        "name": "dynamic_quantization",
        "config": {"quantizer": "torch_dynamic"},
        "needs_cuda": False,
        "description": "Dynamic INT8 quantization at runtime (CPU-friendly)",
    },
    {
        "name": "llm_int8_quantization",
        "config": {"quantizer": "llm_int8"},
        "needs_cuda": True,
        "description": "Mixed-precision INT8 with outlier handling (needs CUDA + bitsandbytes)",
    },
    {
        "name": "structured_pruning",
        "config": {"pruner": "torch_structured"},
        "needs_cuda": False,
        "description": "Structured pruning (removes whole neurons/channels)",
    },
]

## flag which are runnable on the current device so Step 4 can skip GPU-only ones on CPU
print(f"Device: {device}")
for s in compression_strategies:
    runnable = device == "cuda" or not s["needs_cuda"]
    flag = "runnable" if runnable else "SKIP (GPU-only)"
    print(f"  {s['name']:24s} {s['config']}  ->  {flag}")

## Step 4 — Apply each strategy and measure

Loop the strategies: `deepcopy` the teacher, `smash()` it, benchmark with the evaluation function created earlier. Save a copy of the results as well.

In [ ]:
from pruna import SmashConfig, smash

def extract_model(pruna_model):
    # pull the underlying torch model out of the PrunaModel wrapper
    if hasattr(pruna_model, "model"):
        return pruna_model.model
    if hasattr(pruna_model, "_model"):
        return pruna_model._model
    return pruna_model

for s in compression_strategies:
    name = s["name"]

    # skip GPU-only strategies when no CUDA (e.g. llm_int8 on CPU)
    if s["needs_cuda"] and device != "cuda":
        print(f"--- {name}: SKIPPED (needs CUDA, device={device})\n")
        continue

    print(f"--- {name}: {s['description']}")
    try:
        smash_config = SmashConfig(batch_size=BATCH_SIZE, device=device)
        for algo_type, algo_name in s["config"].items():
            smash_config[algo_type] = algo_name

        model_copy = copy.deepcopy(teacher)          # always compress a copy
        compressed_pruna_model = smash(model=model_copy, smash_config=smash_config)
        compressed_model = extract_model(compressed_pruna_model)
        compressed_model.to(device).eval()

        evaluate_variant(compressed_model, name)      # benchmark + record + save
    except Exception as e:
        print(f"    FAILED: {type(e).__name__}: {e}")
    finally:
        # free memory before the next strategy 
        for obj in ("compressed_pruna_model", "compressed_model", "model_copy"):
            if obj in globals():
                del globals()[obj]
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    print()

print(f"Done. Variants recorded: {list(results.keys())}")

## Complement Quantization + Distillation

Teacher = fine-tuned `bert-base-uncased` (~110M). 

Student = **`distilbert-base-uncased`** (~66M).

*Rationale:* For the smaller model, the distilbert is about half the size as the one we've trained in the other sections, from the same "family" of models and trained on similar vocab and pretrained context.

In [ ]:
from transformers import AutoConfig

STUDENT_NAME = "distilbert-base-uncased"

# student config with the same 5-class head
student_config = AutoConfig.from_pretrained(STUDENT_NAME, num_labels=num_labels)

## param counts: teacher from the saved baseline, student from a temp instantiation
teacher_params = baseline_metrics["n_params"]
_student = AutoModelForSequenceClassification.from_pretrained(STUDENT_NAME, num_labels=num_labels)
student_params = count_params(_student)
del _student; gc.collect()

print(f"Teacher : bert-base-uncased     | {teacher_params:,} params")
print(f"Student : {STUDENT_NAME} | {student_params:,} params")
print(f"Compression ratio (params)      : {teacher_params / student_params:.2f}x smaller")

In [ ]:
from tqdm.auto import tqdm

LOGITS_PATH = MODELS_DIR / "teacher_logits.pt"
RECOMPUTE_LOGITS = False     # set True to force a fresh teacher pass

# training set tokenized in the SAME order as train_texts/train_labels (so logits align with labels)
train_ds = build_eval_dataset(train_texts, train_labels)

if LOGITS_PATH.exists() and not RECOMPUTE_LOGITS:
    print(f"Loading cached teacher logits from {LOGITS_PATH}")
    cached_teacher = torch.load(LOGITS_PATH)
    teacher_logits = cached_teacher["logits"]
else:
    print("Computing teacher soft labels over the training set...")
    teacher.to(device).eval()
    chunks = []
    with torch.no_grad():
        for i in tqdm(range(0, len(train_ds), BATCH_SIZE)):
            batch = train_ds[i:i + BATCH_SIZE]
            inputs = {
                "input_ids": batch["input_ids"].to(device, dtype=torch.long),
                "attention_mask": batch["attention_mask"].to(device, dtype=torch.long),
            }
            chunks.append(teacher(**inputs).logits.cpu())
    teacher_logits = torch.cat(chunks, dim=0)
    torch.save(
        {"logits": teacher_logits,
         "labels": torch.tensor(train_labels),
         "student_name": STUDENT_NAME,
         "max_length": max_length},
        LOGITS_PATH,
    )
    print(f"Saved {tuple(teacher_logits.shape)} logits to {LOGITS_PATH}")

## free the teacher — reload from TEACHER_DIR later if needed
if "teacher" in globals():
    del teacher
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"teacher_logits: {tuple(teacher_logits.shape)}  (rows = train examples, cols = {num_labels} classes)")

We run the frozen teacher over the training set once and save the **raw logits** to disk - we cache them so that we avoid a teacher forward pass everytime and it survives kernel restarts if model takes too long. Temperature is applied later in the loss, so we store raw (un-softened) logits here.

After saving we **free the teacher** to leave memory for student training; reload from `TEACHER_DIR` if a later cell needs it.

In [ ]:
# train the student and keep track of loss

import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import f1_score

STUDENT_DIR     = MODELS_DIR / "student_distilled"
STUDENT_EPOCHS  = 3        # raise on GPU (e.g. ~5-10)
LR              = 2e-5
T               = 3.0      # distillation temperature
ALPHA           = 0.5      # weight on the soft (teacher-matching) loss, equal trust / neutral to start
RETRAIN_STUDENT = False    # set True to force retraining even if a saved student exists

def distillation_loss(student_logits, teacher_logits, labels, T=T, alpha=ALPHA):
    soft = F.kl_div(
        F.log_softmax(student_logits / T, dim=-1),
        F.softmax(teacher_logits / T, dim=-1),
        reduction="batchmean",
    ) * (T ** 2)
    hard = F.cross_entropy(student_logits, labels)
    return alpha * soft + (1 - alpha) * hard #similar to bayesian learning, imiatate the teacher's distr. and hard, get true label right.

@torch.no_grad()
def student_val_f1(model):
    ## lightweight macro-F1 on the shared test set (no carbon tracker — cheap, per-epoch)
    model.eval()
    preds = []
    for i in range(0, len(test_ds), BATCH_SIZE):
        b = test_ds[i:i + BATCH_SIZE]
        out = model(
            input_ids=b["input_ids"].to(device, dtype=torch.long),
            attention_mask=b["attention_mask"].to(device, dtype=torch.long),
        ).logits
        preds.extend(out.argmax(-1).cpu().numpy())
    return f1_score(valid_labels, preds, average="macro")

if STUDENT_DIR.exists() and not RETRAIN_STUDENT:
    print(f"Loading existing distilled student from {STUDENT_DIR}")
    student = AutoModelForSequenceClassification.from_pretrained(STUDENT_DIR).to(device)
else:
    set_seed(SEED)
    student = AutoModelForSequenceClassification.from_pretrained(
        STUDENT_NAME, num_labels=num_labels
    ).to(device)

    ## pair each training example with its cached teacher logits (Step 7 order)
    train_tensors = TensorDataset(
        train_ds[:]["input_ids"],
        train_ds[:]["attention_mask"],
        torch.tensor(train_labels),
        teacher_logits,
    )
    loader = DataLoader(train_tensors, batch_size=BATCH_SIZE, shuffle=True)
    optimizer = torch.optim.AdamW(student.parameters(), lr=LR)

    best_f1 = -1.0
    for epoch in range(1, STUDENT_EPOCHS + 1):
        student.train()
        running = 0.0
        for input_ids, attn, labels, t_logits in tqdm(loader, desc=f"epoch {epoch}"):
            input_ids = input_ids.to(device, dtype=torch.long)
            attn      = attn.to(device, dtype=torch.long)
            labels    = labels.to(device)
            t_logits  = t_logits.to(device)

            optimizer.zero_grad()
            s_logits = student(input_ids=input_ids, attention_mask=attn).logits
            loss = distillation_loss(s_logits, t_logits, labels)
            loss.backward()
            optimizer.step()
            running += loss.item() * input_ids.size(0)

        train_loss = running / len(train_tensors)
        val_f1 = student_val_f1(student)
        print(f"epoch {epoch}: train_loss={train_loss:.4f}  val_f1_macro={val_f1:.4f}")

        ## keep the best-by-val-F1 checkpoint
        if val_f1 > best_f1:
            best_f1 = val_f1
            student.save_pretrained(STUDENT_DIR)
            tokenizer.save_pretrained(STUDENT_DIR)

    print(f"Best val F1: {best_f1:.4f}  |  saved to {STUDENT_DIR}")
    ## reload the best checkpoint for downstream benchmarking
    student = AutoModelForSequenceClassification.from_pretrained(STUDENT_DIR).to(device)

student.eval()
print(f"Student ready: {STUDENT_NAME} | {count_params(student):,} params")

Combined loss `L = α · soft + (1-α) · hard`:
- **soft** = `KL(softmax(student/T) ‖ softmax(teacher/T)) · T²` — matches the teacher's softened class distribution (the "dark knowledge").
- **hard** = cross-entropy on the true labels.

Each training example is paired with its cached teacher logits from the previous step. We log train loss + val F1 per epoch and save the best checkpoint to `models/student_distilled/`.



## model comparison

In [ ]:
# benchmark the distilled student with the same harness (records to results + saves JSON)
evaluate_variant(student, "distilled_student")

# confirm the full set of variants on the same test set
print("\nVariants recorded for comparison:")
for k in results:
    print(f"  - {k}")

## Step 11 — Parameters and model size on disk

`n_params` and `size_mb` are captured by the harness at eval time (size via `save_pretrained` while the model is alive). Here we surface them with compression ratios vs. the original.

> Note: unstructured pruning *zeros* weights but still stores them densely, so its param count and on-disk size barely move — the gain shows up in sparsity, not file size. Structured pruning and distillation actually shrink the architecture.

In [ ]:
# params + on-disk size per variant, with ratios vs the original
orig_params = results["original"]["n_params"]
orig_size   = results["original"].get("size_mb")

rows = []
for name, m in results.items():
    p  = m["n_params"]
    sz = m.get("size_mb")
    rows.append({
        "Method": name,
        "Parameters": p,
        "Param ratio (vs orig)": round(orig_params / p, 2) if p else None,
        "Size (MB)": round(sz, 1) if sz is not None else None,
        "Size ratio (vs orig)": round(orig_size / sz, 2) if (sz and orig_size) else None,
    })

size_df = pd.DataFrame(rows)
size_df

## Step 12 — Full comparison table & figure

One row per variant across the full metric set (performance × efficiency): accuracy, macro F1/precision/recall, speed, RAM, GPU memory, carbon, params, size. Saved to `results/part4_comparison_table.csv` and visualised as a 2×2 bar chart in `figures/compression_comparison.png`.

In [ ]:
## full comparison table — one row per variant, all harness metrics
def row_from(name, m):
    return {
        "Method": name,
        "Accuracy": m.get("accuracy"),
        "F1 (macro)": m.get("f1_macro"),
        "Precision (macro)": m.get("precision_macro"),
        "Recall (macro)": m.get("recall_macro"),
        "Speed (samples/sec)": m.get("inference_speed (samples/sec)"),
        "RAM peak (MB)": m.get("cpu_memory_peak (MB)"),
        "GPU mem (MB)": m.get("gpu_memory_used (MB)"),
        "CO2 (kg)": m.get("carbon_footprint (kg CO2eq)"),
        "Params": m.get("n_params"),
        "Size (MB)": m.get("size_mb"),
    }

comparison_df = pd.DataFrame([row_from(n, m) for n, m in results.items()])

## drop the GPU column on a CPU run (nothing recorded it)
if comparison_df["GPU mem (MB)"].isna().all():
    comparison_df = comparison_df.drop(columns=["GPU mem (MB)"])

comparison_df.to_csv(RESULTS_DIR / "part4_comparison_table.csv", index=False)
print(f"Saved table to {RESULTS_DIR / 'part4_comparison_table.csv'}")
comparison_df.round(4)

In [ ]:
## 2x2 bar chart: performance (F1) vs efficiency (speed, size, carbon)
## original drawn in grey as the reference; other variants in blue
methods = comparison_df["Method"].tolist()
colors  = ["grey" if m == "original" else "steelblue" for m in methods]

panels = [
    ("F1 (macro)",          "F1 (macro) — higher is better"),
    ("Speed (samples/sec)", "Inference speed — higher is better"),
    ("Size (MB)",           "Model size on disk — lower is better"),
    ("CO2 (kg)",            "Carbon per eval run — lower is better"),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (col, title) in zip(axes.ravel(), panels):
    ax.bar(methods, comparison_df[col], color=colors)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=45)
    for lbl in ax.get_xticklabels():
        lbl.set_ha("right")

fig.suptitle("Part 4 — Compression strategy comparison", fontsize=14)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "compression_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved figure to {FIGURES_DIR / 'compression_comparison.png'}")